# Assignment 2 — Fine-tuning `bert-base-multilingual-cased`

**Task:** Binary text classification — predict `is_safe` (safe vs unsafe AI-generated response)  
**Dataset:** Salamandra Guard (same splits as Assignment 1)  
**Model:** `bert-base-multilingual-cased`  
**Text input:** `response` column (raw, minimal cleaning — no lemmatisation, no stopword removal)  

Results are saved to `results/transformer_results.csv` in the same schema as `model_results.csv`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# vê o que existe no teu MyDrive
!ls "/content/drive/MyDrive"

'1º Deliverable AGE-i-TANK.gform'
'1º Deliverable AGE-i-TANK (Respostas).gsheet'
'2ª Edição Inscrição - AGE-i-TANK (Respostas).gsheet'
'2º deliverable.gform'
'3º deliverable.gform'
'AGE-i-TANK | Submissão de documentos (File responses)'
 ceuzinho.psd
'Colab Notebooks'
'Diogo Carvalho Viana.pdf'
'Documento sem nome.gdoc'
'Excel 01-04-21.mp4'
'Excel 18-03-21.mkv'
'Excel 22-04-21.mp4'
'Excel 25-03-21.mkv'
'Excel 29-04-21.mp4'
'Excel extra 14-04-21.mp4'
'experiencia .gdoc'
'Folha de cálculo sem nome.gsheet'
'Formulário sem título (File responses)'
'Formulário sem título (File responses) (1)'
'Formulário sem título (File responses) (2)'
'Guardado a partir do Chrome'
'Guardado a partir do Chrome (1)'
 kika1.jpg
'Lot Sizing Techniques and Manufacturing  Resource Planning (2).pdf'
'Materials Requirement Planning (1).pdf'
'Operations Scheduling (1).pdf'
'Photoshop CS6 Portable.zip (Unzipped Files)'
 PLN
 received_1123089781147420.jpeg
 received_1123089801147418~3~2.jpg
 Screenshot_2

In [3]:
os.chdir("/content/drive/MyDrive/PLN")
!pwd
!ls

/content/drive/MyDrive/PLN
models	results  TREINO1


In [4]:
import os

os.chdir("/content/drive/MyDrive/PLN/TREINO1")
!pwd
!ls

/content/drive/MyDrive/PLN/TREINO1
data  models  notebooks  README.md  results


In [5]:
!ls notebooks
!ls data

08_Transformers_BERT.ipynb
features  processed  raw


## 0. Install dependencies
Run once, then restart the kernel.

In [6]:
# Uncomment to install
!pip install transformers datasets evaluate accelerate scikit-learn pandas numpy torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


## 1. Imports

In [7]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import Dataset
import evaluate

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

PyTorch: 2.10.0+cu128
CUDA available: True
Device: cuda


## 2. Configuration

In [8]:
# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
set_seed(SEED)

# ── Paths ────────────────────────────────────────────────────────
BASE_DIR    = Path(".")
DATA_RAW    = BASE_DIR / "data" / "raw"
RESULTS_DIR = BASE_DIR / "results"
MODEL_DIR   = BASE_DIR / "models" / "bert-multilingual"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────
MODEL_NAME   = "bert-base-multilingual-cased"
DATASET_TAG  = "response_raw_max384"          # label for results CSV

# ── Hyperparameters ──────────────────────────────────────────────
MAX_LENGTH   = 384 # Cada texto é truncado/padded para 256 tokens. Bom porque: reduz memória; acelera treino. Possível problema: Se houver textos muito longos, parte da informação pode ser cortada. Mas 256 costuma ser razoável.
BATCH_SIZE   = 16 #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
GRAD_ACCUM   = 2                       # effective batch = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.1
VAL_SIZE      = 0.1                    # 10 % of training set → validation

print(f"Model : {MODEL_NAME}")
print(f"Max length : {MAX_LENGTH}")
print(f"Batch size (per device): {BATCH_SIZE}  |  grad_accum: {GRAD_ACCUM}")
print(f"Epochs : {NUM_EPOCHS}  |  LR : {LEARNING_RATE}")

Model : bert-base-multilingual-cased
Max length : 384
Batch size (per device): 16  |  grad_accum: 2
Epochs : 3  |  LR : 2e-05


## 3. Load dataset

In [9]:
train_path = DATA_RAW / "train_raw.csv"
test_path  = DATA_RAW / "test_raw.csv"

df_train_full = pd.read_csv(train_path)
df_test       = pd.read_csv(test_path)

print(f"Train raw : {len(df_train_full):,} rows  |  columns: {list(df_train_full.columns)}")
print(f"Test  raw : {len(df_test):,} rows")
print("\nClass distribution in train (is_safe):")
print(df_train_full["is_safe"].value_counts())
print("\nClass distribution in test (is_safe):")
print(df_test["is_safe"].value_counts())

Train raw : 20,316 rows  |  columns: ['id', 'prompt', 'response', 'language', 'is_safe', 's_codes', 'majority_vote', 'majority_c_cat', 'Annotator_1', 'Annotator_2', 'Annotator_3', 'GPT_4o_LABEL_RESPONSE', 'GPT_OSS_LABEL_RESPONSE', 'Nemotron_label', 'nemo_label_og', 'prompt_length', 'response_length']
Test  raw : 1,006 rows

Class distribution in train (is_safe):
is_safe
True     10871
False     9445
Name: count, dtype: int64

Class distribution in test (is_safe):
is_safe
False    554
True     452
Name: count, dtype: int64


## 4. Minimal preprocessing

No lemmatisation, no stopword removal — transformers work on raw text.
We only fix: nulls, duplicated whitespace, leading/trailing spaces.

In [10]:
def minimal_clean(text: str) -> str:
    """Strip, collapse whitespace, remove null bytes."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", "")
    text = re.sub(r"[\t\r\f]+", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["response"].apply(minimal_clean)
    # Encode label: True/1 → 1 (safe), False/0 → 0 (unsafe)
    df["label"] = df["is_safe"].map(
        lambda v: 1 if str(v).strip().lower() in {"true", "1"} else 0
    )
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"] != ""]
    return df[["text", "label", "language"]]


df_train_full = prepare_df(df_train_full)
df_test       = prepare_df(df_test)

print(f"After cleaning — train: {len(df_train_full):,}  |  test: {len(df_test):,}")
print("\nLabel distribution train:")
print(df_train_full["label"].value_counts())
print("\nLabel distribution test:")
print(df_test["label"].value_counts())

After cleaning — train: 20,310  |  test: 1,003

Label distribution train:
label
1    10865
0     9445
Name: count, dtype: int64

Label distribution test:
label
0    554
1    449
Name: count, dtype: int64


## 5. Train / Validation split

In [11]:
df_train, df_val = train_test_split(
    df_train_full,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df_train_full["label"],
)

print(f"Train : {len(df_train):,}  |  Val : {len(df_val):,}  |  Test : {len(df_test):,}")
print(f"\nTrain label balance : {df_train['label'].mean():.3f} (fraction safe)")
print(f"Val   label balance : {df_val['label'].mean():.3f}")
print(f"Test  label balance : {df_test['label'].mean():.3f}")

Train : 18,279  |  Val : 2,031  |  Test : 1,003

Train label balance : 0.535 (fraction safe)
Val   label balance : 0.535
Test  label balance : 0.448


## 6. Tokenisation

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

# Token length stats to check if MAX_LENGTH is appropriate
sample_lengths = [
    len(tokenizer(t, truncation=False)["input_ids"])
    for t in df_train["text"].sample(500, random_state=SEED)
]
print(f"Token length (sample n=500):")
print(f"  Mean   : {np.mean(sample_lengths):.1f}")
print(f"  Median : {np.median(sample_lengths):.1f}")
print(f"  P95    : {np.percentile(sample_lengths, 95):.1f}")
print(f"  Max    : {np.max(sample_lengths)}")
print(f"  MAX_LENGTH={MAX_LENGTH} covers ~{np.mean(np.array(sample_lengths) <= MAX_LENGTH)*100:.1f}% of samples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


Token length (sample n=500):
  Mean   : 216.6
  Median : 180.0
  P95    : 547.0
  Max    : 812
  MAX_LENGTH=384 covers ~82.2% of samples


## 7. Build HuggingFace Datasets

In [13]:
def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True, batch_size=256)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


print("Tokenising train set ...")
hf_train = df_to_hf_dataset(df_train)
print("Tokenising validation set ...")
hf_val   = df_to_hf_dataset(df_val)
print("Tokenising test set ...")
hf_test  = df_to_hf_dataset(df_test)

print(f"\nhf_train features : {hf_train.features}")

Tokenising train set ...


Map:   0%|          | 0/18279 [00:00<?, ? examples/s]

Tokenising validation set ...


Map:   0%|          | 0/2031 [00:00<?, ? examples/s]

Tokenising test set ...


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]


hf_train features : {'text': Value('string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


## 8. Load model

In [14]:
id2label = {0: "unsafe", 1: "safe"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params    : 177,854,978
Trainable params: 177,854,978


## 9. Metrics

In [15]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc    = accuracy_score(labels, preds)
    prec   = precision_score(labels, preds, average="macro", zero_division=0)
    rec    = recall_score(labels, preds, average="macro", zero_division=0)
    f1mac  = f1_score(labels, preds, average="macro", zero_division=0)
    f1wei  = f1_score(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy" : acc,
        "precision": prec,
        "recall"   : rec,
        "f1_macro" : f1mac,
        "f1_weighted": f1wei,
    }

## 10. Training

In [16]:
import math

# Compute warmup_steps explicitly (replaces deprecated warmup_ratio)
steps_per_epoch = math.ceil(len(df_train) / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(WARMUP_RATIO * total_steps)
print(f"steps_per_epoch={steps_per_epoch}  total_steps={total_steps}  warmup_steps={warmup_steps}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,         # replaces deprecated warmup_ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=tokenizer,        # replaces deprecated tokenizer= arg
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training ...")
train_result = trainer.train()

print("\nTraining summary:")
print(train_result.metrics)

steps_per_epoch=572  total_steps=1716  warmup_steps=171
Starting training ...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro,F1 Weighted
1,0.784643,0.368464,0.832595,0.832758,0.830410,0.831255,0.832299
2,0.586569,0.330292,0.869522,0.871372,0.872634,0.869490,0.869633
3,0.374588,0.325249,0.889217,0.888833,0.888440,0.888626,0.889189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Training summary:
{'train_runtime': 1269.9581, 'train_samples_per_second': 43.18, 'train_steps_per_second': 1.351, 'total_flos': 1.082116570708224e+16, 'train_loss': 0.6679699712699944, 'epoch': 3.0}


## 11. Evaluation on the test set

In [17]:
# Full evaluation with HF Trainer (uses compute_metrics)
eval_output = trainer.evaluate(hf_test)
print("Test metrics (HF Trainer):")
for k, v in eval_output.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Test metrics (HF Trainer):
  eval_loss: 0.7749
  eval_accuracy: 0.7637
  eval_precision: 0.7629
  eval_recall: 0.7570
  eval_f1_macro: 0.7587
  eval_f1_weighted: 0.7624
  eval_runtime: 5.7492
  eval_samples_per_second: 174.4600
  eval_steps_per_second: 5.5660
  epoch: 3.0000


In [18]:
# Collect raw predictions for a detailed sklearn report
preds_output = trainer.predict(hf_test)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(df_test["label"].values)

print("=" * 55)
print("Classification Report (test set)")
print("=" * 55)
print(classification_report(y_true, y_pred, target_names=["unsafe (0)", "safe (1)"]))

print("Confusion Matrix (rows=true, cols=pred):")
cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=["unsafe", "safe"], columns=["pred_unsafe", "pred_safe"]))

Classification Report (test set)
              precision    recall  f1-score   support

  unsafe (0)       0.77      0.82      0.79       554
    safe (1)       0.76      0.69      0.72       449

    accuracy                           0.76      1003
   macro avg       0.76      0.76      0.76      1003
weighted avg       0.76      0.76      0.76      1003

Confusion Matrix (rows=true, cols=pred):
        pred_unsafe  pred_safe
unsafe          455         99
safe            138        311


### Breakdown by language

In [23]:
df_results = df_test.copy().reset_index(drop=True)
df_results["pred"] = y_pred

for lang, grp in df_results.groupby("language"):
    f1 = f1_score(grp["label"], grp["pred"], average="macro", zero_division=0)
    acc = accuracy_score(grp["label"], grp["pred"])
    print(f"Language={lang:4s}  n={len(grp):4d}  acc={acc:.3f}  macro-F1={f1:.3f}")

Language=ca    n= 503  acc=0.773  macro-F1=0.767
Language=es    n= 500  acc=0.754  macro-F1=0.750


## 12. Save results

In [20]:
acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec   = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1mac = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1wei = f1_score(y_true, y_pred, average="weighted", zero_division=0)

new_row = pd.DataFrame([{
    "dataset"    : DATASET_TAG,
    "model"      : MODEL_NAME,
    "accuracy"   : acc,
    "precision"  : prec,
    "recall"     : rec,
    "f1_macro"   : f1mac,
    "f1_weighted": f1wei,
}])

out_path = RESULTS_DIR / "transformer_results.csv"
if out_path.exists():
    existing = pd.read_csv(out_path)
    # Remove previous run for same model+dataset if any
    existing = existing[
        ~((existing["model"] == MODEL_NAME) & (existing["dataset"] == DATASET_TAG))
    ]
    df_out = pd.concat([existing, new_row], ignore_index=True)
else:
    df_out = new_row

df_out.to_csv(out_path, index=False)
print(f"Results saved to {out_path}")
print()
print(new_row.to_string(index=False))

Results saved to results/transformer_results.csv

            dataset                        model  accuracy  precision   recall  f1_macro  f1_weighted
response_raw_max384 bert-base-multilingual-cased  0.763709   0.762911 0.756975  0.758736     0.762362


## 13. Comparison with Assignment 1 baselines

In [21]:
baseline_path = RESULTS_DIR / "model_results.csv"
df_baseline = pd.read_csv(baseline_path)

# Keep only the best classical model per dataset variant
best_classical = (
    df_baseline
    .sort_values("f1_macro", ascending=False)
    .drop_duplicates(subset=["dataset"])
    .head(5)
    [["dataset", "model", "accuracy", "precision", "recall", "f1_macro"]]
)

transformer_row = new_row[["dataset", "model", "accuracy", "precision", "recall", "f1_macro"]]

df_compare = pd.concat([best_classical, transformer_row], ignore_index=True)

print("=" * 80)
print("Comparison: Classical (best per variant) vs bert-base-multilingual-cased")
print("=" * 80)
print(df_compare.to_string(index=False))

Comparison: Classical (best per variant) vs bert-base-multilingual-cased
            dataset                        model  accuracy  precision   recall  f1_macro
      resp_no_punct          Logistic Regression  0.791626   0.790617 0.791303  0.790883
      comb_no_punct          Logistic Regression  0.791626   0.790668 0.790375  0.790512
    comb_with_punct          Logistic Regression  0.789409   0.788385 0.788405  0.788395
    resp_with_punct          Logistic Regression  0.788916   0.787898 0.788563  0.788157
response_raw_max384 bert-base-multilingual-cased  0.763709   0.762911 0.756975  0.758736


## 14. Save fine-tuned model

In [22]:
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Model saved to {MODEL_DIR / 'final'}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to models/bert-multilingual/final


## Summary

| Step | Details |
|---|---|
| Model | `bert-base-multilingual-cased` |
| Text input | `response` column (raw, minimal cleaning) |
| Train / Val / Test | ~18 k / ~2 k / 1 k |
| Max sequence length | 256 tokens |
| Epochs | 3 (early stopping patience=2) |
| Learning rate | 2e-5 |
| Batch size (effective) | 32 |
| Best model criterion | macro-F1 on validation |

Results saved to `results/transformer_results.csv`.  
Fine-tuned weights saved under `models/bert-multilingual/final/`.